# 00 · Pipeline completo — RV32I_Zmmul_Xicrc

Este notebook executa, na ordem, **todo** o fluxo de engenharia do processador
submetido à Fase 2 da ChampionCHIP eXperience e termina no painel de resultados:

| etapa | o que prova | script |
|---|---|---|
| 1. Lint | o RTL é sintetizável (Verilator `-Wall`) | `scripts/run_lint.sh` |
| 2. Regressão | 8 testes unitários + 4 de ISA + firmware + 20 programas aleatórios vs modelo de referência | `scripts/run_regression.sh` |
| 3. Mutação | a verificação realmente detecta bugs injetados | `scripts/run_mutation.sh` |
| 4. Gate-level | a netlist pós-layout se comporta igual ao modelo | `scripts/run_gls.sh` |
| 5. Relatórios | agrega todas as evidências | `tools/build_reports.py` |
| 6. Dashboard | painel HTML autocontido | `tools/build_dashboard.py` |

O fluxo físico (OpenLane, ~50 min por rodada) fica no notebook `03`. Os detalhes
de cada etapa estão nos notebooks `01` a `03`.

> **Pré-requisitos:** Docker em execução com a imagem `championchip-dev`
> (`make build-image`) — ou as ferramentas instaladas localmente com
> `CHAMPIONCHIP_NATIVE=1`. Ver `README.md`.

In [1]:
import sys, json
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))          # notebooks/_common.py
from _common import ROOT, run, py, summary, style, SERIES
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import SVG, Image, Markdown, display
print("projeto:", ROOT.name)

projeto: ChampionCHIP (Terminar)


## 1. Lint sintetizável

In [2]:
run("bash scripts/run_lint.sh", tail=5)

lint: 17 warnings, 0 erros (detalhes em docs/evidence/logs/lint.log)
[exit code 0]


0

## 2. Regressão completa

In [3]:
run("bash scripts/run_regression.sh", tail=30)

==== 1. TESTES UNITARIOS ====
PASS  unit/tb_alu
PASS  unit/tb_regfile
PASS  unit/tb_imm_gen
PASS  unit/tb_branch_cmp
PASS  unit/tb_mult_unit
PASS  unit/tb_lsu
PASS  unit/tb_address_decoder
PASS  unit/tb_crc_unit
==== 2. TESTES DE ISA (dirigidos por instrucao) ====
PASS  isa/tb_isa_alu
PASS  isa/tb_isa_mem_upper_sys
PASS  isa/tb_isa_branch_jump
PASS  isa/tb_isa_zmmul
==== 3. FIRMWARE (GCC RISC-V -> ELF -> hex -> simulacao) ====
PASS  firmware/smoke_test
==== 4. DIFERENCIAL RANDOMIZADO vs modelo de referencia (20 x 200 instr.) ====
cobertura do corpo aleatorio: 40 instrucoes distintas, 3644 execucoes
random: 20/20 programas identicos ao modelo de referencia
PASS  random/diferencial (20/20)
==== RESULTADO: TUDO PASSOU ====
[exit code 0]


0

## 3. Teste de mutação

In [4]:
run("bash scripts/run_mutation.sh", tail=16)

MORTO      ALU: SRA vira SRL                        detectado em  8/20 programas
MORTO      ALU: SLT vira SLTU                       detectado em 14/20 programas
MORTO      ALU: SUB vira ADD                        detectado em 12/20 programas
MORTO      MUL: MULHSU usa signed x signed          detectado em  4/20 programas
MORTO      MUL: MULHU usa signed x signed           detectado em  7/20 programas
MORTO      LSU: LB sem sign-extend                  detectado em 18/20 programas
MORTO      LSU: LH sem sign-extend                  detectado em 15/20 programas
MORTO      LSU: SH sempre na metade baixa           detectado em 18/20 programas
MORTO      BRANCH: BLTU vira BLT                    detectado em 11/20 programas
MORTO      BRANCH: BGE vira BGT                     detectado em  4/20 programas
MORTO      IMM: imediato B sem bit 11               detectado em 19/20 programas
MORTO      REGFILE: x0 gravavel                     detectado em 18/20 programas
MORTO      CORE: link do JAL

0

## 4. Simulação gate-level da netlist pós-layout

In [5]:
run("bash scripts/run_gls.sh", tail=10)

netlist: docs/evidence/openlane/run_best/rv32_core.nl.v (5.9M)
==== GL 1. firmware de smoke-test sobre a netlist ====
PASS  gl/firmware
==== GL 2. diferencial randomizado sobre a netlist (10 programas) ====
PASS  gl/random (10/10 programas identicos ao modelo de referencia)
==== GL RESULTADO: NETLIST EQUIVALENTE AO MODELO ====
[exit code 0]


0

## 5. Resumo consolidado

In [6]:
S = summary()
v = S["verification"]
display(Markdown(f"""
| indicador | resultado |
|---|---|
| Instruções fechadas | **{S['isa']['closed']}/{S['isa']['total']}** (RV32I {S['isa']['rv32i']}/40 · Zmmul {S['isa']['zmmul']}/4 · Xicrc bloqueada) |
| Suítes de regressão | **{v['regression_pass']}/{len(v['regression'])}** PASS |
| Mutation score | **{v['mutation_killed']}/{len(v['mutation'])}** bugs detectados |
| Gate-level | {', '.join(r['teste'] + ' ' + r['status'] for r in v['gls']) or '—'} |
"""))
pd.DataFrame(S["physical"])[["label", "clock_period_ns", "clock_mhz", "setup_worst_slack_ns", "fmax_mhz_worst_corner",
                             "drc_klayout", "lvs_errors", "antenna_violations", "stdcell_area_um2", "power_total_w"]]

reports/summary.json: ISA 44/47, 14 suites, 13 mutantes, 3 runs fisicos




| indicador | resultado |
|---|---|
| Instruções fechadas | **44/47** (RV32I 40/40 · Zmmul 4/4 · Xicrc bloqueada) |
| Suítes de regressão | **14/14** PASS |
| Mutation score | **13/13** bugs detectados |
| Gate-level | firmware PASS, random 10 programas PASS |


,label,clock_period_ns,clock_mhz,setup_worst_slack_ns,fmax_mhz_worst_corner,drc_klayout,lvs_errors,antenna_violations,stdcell_area_um2,power_total_w
0,baseline,40.0,25.0,11.568808,35.2,0,0,2,259760,0.031925
1,opt30,30.0,33.3,1.144826,34.7,0,0,2,259752,0.042649
2,final,30.0,33.3,2.641659,36.6,0,0,19,220016,0.042005


## 6. Dashboard

In [7]:
py("tools/build_dashboard.py")
print("Abra no navegador: dashboard/index.html")

dashboard\index.html: 1048 KB

Abra no navegador: dashboard/index.html
